# Empirical Analysis of Network Isomorphisms and Symmetry Breaking

## Overview

Fefferman's theorem tells us that identifiable networks are unique **up to symmetry transformations** (permutations and, for odd activations, sign flips). This notebook investigates these symmetries empirically:

1. **Network Isomorphisms**: Given two networks, can we detect whether they compute the same function? We distinguish:
   - **Extensional isomorphism**: Same input-output mapping (black-box equivalence)
   - **Faithful isomorphism**: Parameters related by permutations and sign flips (structural equivalence)

2. **Symmetry Breaking**: Strategies to reduce the number of equivalent parameterizations, pushing toward a **canonical form**:
   - Weight-norm ordering
   - L1/L2 regularization
   - Distinct bias constraints
   - Anti-clone regularization

### Theoretical Context

For a tanh network with hidden layer sizes $(n_1, n_2, \ldots, n_L)$, the number of equivalent parameterizations is:
$$|\text{Equiv}| = \prod_{l=1}^{L} n_l! \cdot 2^{n_l}$$

Symmetry-breaking constraints aim to reduce this combinatorial explosion to ideally a **single** canonical representative.

### References
- Fefferman, C. (1994). *Reconstructing a Neural Net from its Output*.
- Bona-Pellissier, Miche, Malgouyres (2022). *Parameter Identifiability of Neural Networks*.
- Petzka, H., Trimmel, M. (2020). *On the Identifiability of Neural Networks*.

## 1. Imports and Setup

In [ ]:
import sys
import os
import copy
from itertools import combinations

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment

# Add src/ to path for project module imports
sys.path.insert(0, '..')
from src.identifiability_checks import (
    build_network,
    extract_parameters,
    check_no_clones,
    check_non_degeneracy,
    identifiability_report,
)
from src.network_isomorphisms import (
    apply_permutation,
    apply_sign_flips,
    check_extensional_isomorphism,
    check_faithful_isomorphism,
    find_layer_permutation,
    create_permuted_network,
)
from src.symmetry_breaking import (
    ordered_initialization,
    OrderingRegularizer,
    DistinctBiasRegularizer,
    AntiCloneRegularizer,
    train_with_symmetry_breaking,
)

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

print(f"PyTorch version: {torch.__version__}")
print(f"NumPy version:   {np.__version__}")

## 2. Generating Networks with Known Isomorphisms

To test our isomorphism detection, we create network pairs where the ground-truth relationship is known:
- **Model A**: Original randomly initialized network
- **Model B**: Created from Model A by randomly permuting hidden neurons and flipping signs (known isomorphism)
- **Model C**: Independently initialized network (should NOT be isomorphic)

In [ ]:
architecture = [5, 10, 8, 1]
activation = 'tanh'

# Model A: original network
torch.manual_seed(42)
model_A = build_network(architecture, activation)

# Model B: isomorphic copy (permuted + sign-flipped)
torch.manual_seed(99)  # Different seed for random permutations/sign flips
model_B = create_permuted_network(model_A, architecture, activation)

# Model C: completely different network
torch.manual_seed(123)
model_C = build_network(architecture, activation)

print(f"Architecture: {architecture}")
print(f"Activation:   {activation}")
print(f"Parameters:   {sum(p.numel() for p in model_A.parameters()):,}")
print()
print("Model A: Original network (seed=42)")
print("Model B: Isomorphic copy of A (permuted + sign-flipped)")
print("Model C: Independent network (seed=123)")

In [ ]:
# Verify that A and B produce the same outputs
X_test = torch.randn(500, architecture[0])

model_A.eval()
model_B.eval()
model_C.eval()

with torch.no_grad():
    y_A = model_A(X_test)
    y_B = model_B(X_test)
    y_C = model_C(X_test)

print("Output comparison (max absolute difference):")
print(f"  |A - B| max: {(y_A - y_B).abs().max().item():.2e}  (should be ~0)")
print(f"  |A - C| max: {(y_A - y_C).abs().max().item():.2e}  (should be large)")
print()

# Scatter plot of outputs
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_A.numpy(), y_B.numpy(), alpha=0.5, s=10, c='#3498db')
lim = [min(y_A.min().item(), y_B.min().item()), max(y_A.max().item(), y_B.max().item())]
axes[0].plot(lim, lim, 'r--', linewidth=1.5, label='y = x')
axes[0].set_xlabel('Model A Output')
axes[0].set_ylabel('Model B Output')
axes[0].set_title('A vs B (Isomorphic Pair)', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].scatter(y_A.numpy(), y_C.numpy(), alpha=0.5, s=10, c='#e74c3c')
axes[1].set_xlabel('Model A Output')
axes[1].set_ylabel('Model C Output')
axes[1].set_title('A vs C (Non-Isomorphic)', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.suptitle('Network Output Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Detecting Faithful Isomorphisms

A **faithful isomorphism** between networks A and B means there exist:
- Permutation matrices $P_l$ for each hidden layer $l$
- Sign-flip diagonal matrices $S_l = \text{diag}(\pm 1, \ldots, \pm 1)$ (for tanh)

such that:
$$W^{(l)}_B = S_l P_l W^{(l)}_A, \quad b^{(l)}_B = S_l P_l b^{(l)}_A$$
$$W^{(l+1)}_B = W^{(l+1)}_A P_l^T S_l$$

We use the **Hungarian algorithm** on a cost matrix of neuron-to-neuron distances to find the optimal alignment.

In [ ]:
# Extract parameters
layers_A = extract_parameters(model_A)
layers_B = extract_parameters(model_B)
layers_C = extract_parameters(model_C)

# Test faithful isomorphism: A vs B (should succeed)
print("=" * 60)
print("FAITHFUL ISOMORPHISM: Model A vs Model B")
print("=" * 60)
faith_AB = check_faithful_isomorphism(layers_A, layers_B, allow_sign_flips=True)
print(f"Is faithfully isomorphic: {faith_AB['is_faithfully_isomorphic']}")
print(f"Total alignment cost:     {faith_AB['total_alignment_cost']:.6f}")
if faith_AB['is_faithfully_isomorphic']:
    for l, (perm, signs) in enumerate(zip(faith_AB['permutations'], faith_AB['signs'])):
        print(f"  Layer {l}: permutation={perm}")
        n_flips = sum(1 for s in signs if s == -1)
        print(f"           sign flips: {n_flips}/{len(signs)} neurons flipped")

In [ ]:
# Test faithful isomorphism: A vs C (should fail)
print("=" * 60)
print("FAITHFUL ISOMORPHISM: Model A vs Model C")
print("=" * 60)
faith_AC = check_faithful_isomorphism(layers_A, layers_C, allow_sign_flips=True)
print(f"Is faithfully isomorphic: {faith_AC['is_faithfully_isomorphic']}")
print(f"Total alignment cost:     {faith_AC['total_alignment_cost']:.6f}")

In [ ]:
# Visualize alignment costs as heatmaps for each hidden layer
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

pairs = [
    ('A vs B (Isomorphic)', layers_A, layers_B),
    ('A vs C (Non-Isomorphic)', layers_A, layers_C),
]

for row, (pair_name, la, lb) in enumerate(pairs):
    for col in range(min(2, len(la) - 1)):  # Hidden layers
        W_a, b_a = la[col]['weight'], la[col]['bias']
        W_b, b_b = lb[col]['weight'], lb[col]['bias']
        D = W_a.shape[0]
        
        cost_matrix = torch.zeros(D, D)
        for i in range(D):
            for j in range(D):
                diff_pos = (W_a[i] - W_b[j]).norm() + abs(b_a[i] - b_b[j])
                diff_neg = (W_a[i] + W_b[j]).norm() + abs(b_a[i] + b_b[j])
                cost_matrix[i, j] = min(diff_pos.item(), diff_neg.item())
        
        im = axes[row, col].imshow(cost_matrix.numpy(), cmap='viridis', aspect='auto')
        axes[row, col].set_title(f'{pair_name}\nLayer {col} Cost Matrix', fontsize=11)
        axes[row, col].set_xlabel('Neuron in Network 2')
        axes[row, col].set_ylabel('Neuron in Network 1')
        plt.colorbar(im, ax=axes[row, col], fraction=0.046, pad=0.04)
        
        # Mark optimal assignment
        row_ind, col_ind = linear_sum_assignment(cost_matrix.numpy())
        axes[row, col].scatter(col_ind, row_ind, color='red', marker='x', s=100,
                               linewidths=2, zorder=5, label='Optimal assignment')
        axes[row, col].legend(loc='upper right', fontsize=8)

plt.suptitle('Neuron Alignment Cost Matrices (Hungarian Algorithm)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Detecting Extensional Isomorphisms

An **extensional isomorphism** is a weaker notion: two networks are extensionally isomorphic if they produce the same output for all inputs (up to numerical tolerance). This is a black-box test that does not examine the internal structure.

Every faithful isomorphism implies an extensional isomorphism, but not vice versa (in the non-identifiable case).

In [ ]:
# Extensional isomorphism tests
print("=" * 60)
print("EXTENSIONAL ISOMORPHISM DETECTION")
print("=" * 60)

ext_AB = check_extensional_isomorphism(model_A, model_B, architecture, n_test=2000)
print(f"\nA vs B (isomorphic pair):")
print(f"  Extensionally isomorphic: {ext_AB['is_extensionally_isomorphic']}")
print(f"  Max output difference:    {ext_AB['max_output_difference']:.2e}")
print(f"  Mean output difference:   {ext_AB['mean_output_difference']:.2e}")

ext_AC = check_extensional_isomorphism(model_A, model_C, architecture, n_test=2000)
print(f"\nA vs C (different network):")
print(f"  Extensionally isomorphic: {ext_AC['is_extensionally_isomorphic']}")
print(f"  Max output difference:    {ext_AC['max_output_difference']:.2e}")
print(f"  Mean output difference:   {ext_AC['mean_output_difference']:.2e}")

In [ ]:
# Multiple permuted copies: create several isomorphic versions and test
n_copies = 5
ext_diffs = []
faith_costs = []

print("Creating multiple isomorphic copies of Model A and testing...\n")
print(f"{'Copy':<8} {'Ext. Max Diff':<18} {'Faith. Cost':<16} {'Ext. Iso?':<12} {'Faith. Iso?':<12}")
print("-" * 66)

for i in range(n_copies):
    torch.manual_seed(i * 17 + 7)  # Different permutations each time
    model_copy = create_permuted_network(model_A, architecture, activation)
    
    ext_result = check_extensional_isomorphism(model_A, model_copy, architecture)
    layers_copy = extract_parameters(model_copy)
    faith_result = check_faithful_isomorphism(layers_A, layers_copy)
    
    ext_diffs.append(ext_result['max_output_difference'])
    faith_costs.append(faith_result['total_alignment_cost'])
    
    print(f"  {i:<6} {ext_result['max_output_difference']:<18.2e} "
          f"{faith_result['total_alignment_cost']:<16.6f} "
          f"{'Yes' if ext_result['is_extensionally_isomorphic'] else 'No':<12} "
          f"{'Yes' if faith_result['is_faithfully_isomorphic'] else 'No':<12}")

## 5. Symmetry Breaking Experiments

We now test strategies to **break** the permutation and sign-flip symmetries, aiming to reduce the number of equivalent parameterizations.

### Strategies
1. **Ordered Initialization**: Sort neurons by bias value to fix a canonical ordering
2. **L1/L2 Regularization**: Standard weight decay that indirectly reduces symmetry
3. **Ordering Regularizer**: Penalizes violations of weight-norm ordering
4. **Distinct Bias Regularizer**: Penalizes neurons with similar bias magnitudes
5. **Anti-Clone Regularizer**: Penalizes high cosine similarity between neurons

In [ ]:
# Strategy 1: Ordered Initialization
torch.manual_seed(42)
model_ordered = build_network([5, 10, 10, 1], 'tanh')

print("=== Before Ordered Initialization ===")
ll_before = [m for m in model_ordered.modules() if isinstance(m, nn.Linear)]
print(f"Layer 0 biases: {ll_before[0].bias.data.numpy().round(4)}")

model_ordered = ordered_initialization(model_ordered)

print("\n=== After Ordered Initialization ===")
ll_after = [m for m in model_ordered.modules() if isinstance(m, nn.Linear)]
print(f"Layer 0 biases: {ll_after[0].bias.data.numpy().round(4)}")
print(f"Biases sorted (descending): {all(ll_after[0].bias.data[i] >= ll_after[0].bias.data[i+1] for i in range(9))}")

In [ ]:
# Strategy 2-5: Train with various regularization strategies
# Generate synthetic regression data
torch.manual_seed(42)
train_arch = [2, 8, 8, 1]
X_train = torch.randn(500, train_arch[0])
y_train = torch.sin(X_train[:, 0:1]) + 0.5 * torch.cos(X_train[:, 1:2]) + 0.1 * torch.randn(500, 1)

n_epochs = 300
strategies = {}

# Baseline: no regularization
torch.manual_seed(42)
model_base = build_network(train_arch, 'tanh')
optimizer_base = torch.optim.Adam(model_base.parameters(), lr=0.01)
criterion = nn.MSELoss()
losses_base = []

for epoch in range(n_epochs):
    optimizer_base.zero_grad()
    loss = criterion(model_base(X_train), y_train)
    loss.backward()
    optimizer_base.step()
    losses_base.append(loss.item())

strategies['Baseline'] = {'model': model_base, 'losses': losses_base}
print(f"Baseline - Final loss: {losses_base[-1]:.6f}")

# L1 Regularization
torch.manual_seed(42)
model_l1 = build_network(train_arch, 'tanh')
optimizer_l1 = torch.optim.Adam(model_l1.parameters(), lr=0.01)
losses_l1 = []
l1_weight = 0.001

for epoch in range(n_epochs):
    optimizer_l1.zero_grad()
    loss = criterion(model_l1(X_train), y_train)
    l1_reg = sum(p.abs().sum() for p in model_l1.parameters())
    total_loss = loss + l1_weight * l1_reg
    total_loss.backward()
    optimizer_l1.step()
    losses_l1.append(loss.item())

strategies['L1 Regularization'] = {'model': model_l1, 'losses': losses_l1}
print(f"L1 Reg   - Final loss: {losses_l1[-1]:.6f}")

# L2 Regularization (weight decay)
torch.manual_seed(42)
model_l2 = build_network(train_arch, 'tanh')
optimizer_l2 = torch.optim.Adam(model_l2.parameters(), lr=0.01, weight_decay=0.01)
losses_l2 = []

for epoch in range(n_epochs):
    optimizer_l2.zero_grad()
    loss = criterion(model_l2(X_train), y_train)
    loss.backward()
    optimizer_l2.step()
    losses_l2.append(loss.item())

strategies['L2 Regularization'] = {'model': model_l2, 'losses': losses_l2}
print(f"L2 Reg   - Final loss: {losses_l2[-1]:.6f}")

# Identifiability-aware regularization (ordering + bias + anti-clone)
torch.manual_seed(42)
model_ident = build_network(train_arch, 'tanh')
optimizer_ident = torch.optim.Adam(model_ident.parameters(), lr=0.01)
ordering_reg = OrderingRegularizer(weight=0.05)
bias_reg = DistinctBiasRegularizer(weight=0.03)
clone_reg = AntiCloneRegularizer(weight=0.02)
losses_ident = []

for epoch in range(n_epochs):
    optimizer_ident.zero_grad()
    loss = criterion(model_ident(X_train), y_train)
    total_loss = loss + ordering_reg(model_ident) + bias_reg(model_ident) + clone_reg(model_ident)
    total_loss.backward()
    optimizer_ident.step()
    losses_ident.append(loss.item())

strategies['Identifiability Reg.'] = {'model': model_ident, 'losses': losses_ident}
print(f"Ident Reg - Final loss: {losses_ident[-1]:.6f}")

In [ ]:
# Plot training curves
fig, ax = plt.subplots(figsize=(12, 6))

colors_strat = {
    'Baseline': '#95a5a6',
    'L1 Regularization': '#3498db',
    'L2 Regularization': '#e74c3c',
    'Identifiability Reg.': '#2ecc71',
}

for name, data in strategies.items():
    ax.plot(data['losses'], label=name, linewidth=2, color=colors_strat[name])

ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('MSE Loss', fontsize=12)
ax.set_title('Training Loss: Baseline vs Symmetry-Breaking Strategies', fontsize=14, fontweight='bold')
ax.set_yscale('log')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Visualization: Weight Distributions Before and After Symmetry Breaking

We compare the distribution of weights and biases across strategies to see how regularization affects parameter structure.

In [ ]:
# Weight distribution histograms
fig, axes = plt.subplots(2, len(strategies), figsize=(5 * len(strategies), 10))

for col, (name, data) in enumerate(strategies.items()):
    layers = extract_parameters(data['model'])
    
    # Collect all weights and biases from hidden layers
    all_weights = []
    all_biases = []
    for l in layers[:-1]:  # Hidden layers only
        all_weights.extend(l['weight'].numpy().flatten().tolist())
        all_biases.extend(l['bias'].numpy().tolist())
    
    color = colors_strat[name]
    
    # Weight histogram
    axes[0, col].hist(all_weights, bins=30, color=color, alpha=0.7, edgecolor='black')
    axes[0, col].axvline(0, color='red', linestyle='--', alpha=0.5)
    axes[0, col].set_title(f'{name}\nWeight Distribution', fontsize=11, fontweight='bold')
    axes[0, col].set_xlabel('Weight Value')
    axes[0, col].set_ylabel('Count')
    axes[0, col].grid(True, alpha=0.3)
    
    # Bias histogram
    axes[1, col].hist(all_biases, bins=15, color=color, alpha=0.7, edgecolor='black')
    axes[1, col].axvline(0, color='red', linestyle='--', alpha=0.5)
    axes[1, col].set_title(f'{name}\nBias Distribution', fontsize=11, fontweight='bold')
    axes[1, col].set_xlabel('Bias Value')
    axes[1, col].set_ylabel('Count')
    axes[1, col].grid(True, alpha=0.3)

plt.suptitle('Weight and Bias Distributions Across Strategies',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Neuron similarity heatmaps: compare clone-like structure across strategies
fig, axes = plt.subplots(2, len(strategies), figsize=(5 * len(strategies), 10))

for col, (name, data) in enumerate(strategies.items()):
    layers = extract_parameters(data['model'])
    
    for row in range(min(2, len(layers) - 1)):  # Up to 2 hidden layers
        W = layers[row]['weight']
        W_norm = W / (W.norm(dim=1, keepdim=True) + 1e-8)
        similarity = (W_norm @ W_norm.T).numpy()
        
        im = axes[row, col].imshow(similarity, cmap='RdBu_r', vmin=-1, vmax=1)
        axes[row, col].set_title(f'{name}\nLayer {row} Similarity', fontsize=10)
        axes[row, col].set_xlabel('Neuron')
        axes[row, col].set_ylabel('Neuron')
        plt.colorbar(im, ax=axes[row, col], fraction=0.046, pad=0.04)

plt.suptitle('Neuron Cosine Similarity: Effect of Symmetry-Breaking',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Quantifying Reduction in Equivalence Classes

For a tanh network with layer sizes $(n_1, n_2, \ldots)$, the theoretical number of equivalent parameterizations is $\prod_l n_l! \cdot 2^{n_l}$.

We quantify how symmetry-breaking strategies reduce this by measuring:
1. **Bias distinctness**: fraction of neuron pairs with distinct bias magnitudes
2. **Weight-norm ordering**: whether neurons are sorted by weight norm
3. **Clone-freeness**: absence of near-identical neurons
4. **Effective equivalence class size** (estimated)

In [ ]:
import math

def compute_symmetry_metrics(model, architecture, tol=1e-3):
    """Compute metrics quantifying residual symmetry in a trained network."""
    layers = extract_parameters(model)
    hidden_sizes = architecture[1:-1]
    
    # Theoretical max equivalence classes (tanh)
    theoretical_equiv = 1
    for n in hidden_sizes:
        theoretical_equiv *= math.factorial(n) * (2 ** n)
    
    # Bias distinctness: fraction of pairs with distinct |b_j| values
    total_pairs = 0
    distinct_pairs = 0
    for l in layers[:-1]:
        b = l['bias']
        D = b.shape[0]
        for j, jp in combinations(range(D), 2):
            total_pairs += 1
            if abs(b[j].abs().item() - b[jp].abs().item()) > tol:
                distinct_pairs += 1
    bias_distinctness = distinct_pairs / max(total_pairs, 1)
    
    # Weight-norm ordering: fraction of consecutive pairs in sorted order
    total_consec = 0
    ordered_consec = 0
    for l in layers[:-1]:
        norms = l['weight'].norm(dim=1)
        for j in range(len(norms) - 1):
            total_consec += 1
            if norms[j].item() >= norms[j + 1].item() - tol:
                ordered_consec += 1
    ordering_score = ordered_consec / max(total_consec, 1)
    
    # Clone-freeness
    clones = check_no_clones(layers)
    
    # Max cosine similarity (excluding diagonal)
    max_sim = 0.0
    for l in layers[:-1]:
        W = l['weight']
        W_norm = W / (W.norm(dim=1, keepdim=True) + 1e-8)
        sim = (W_norm @ W_norm.T).numpy()
        np.fill_diagonal(sim, 0)
        max_sim = max(max_sim, np.abs(sim).max())
    
    # Estimated equivalence reduction
    # If biases are all distinct and ordered, permutation symmetry is broken
    perm_factor = 1
    for n in hidden_sizes:
        if ordering_score > 0.9:  # Well-ordered
            perm_factor *= 1  # Symmetry broken
        else:
            perm_factor *= math.factorial(n)
    
    sign_factor = 1
    for n in hidden_sizes:
        if bias_distinctness > 0.95:  # Distinct biases help constrain signs
            sign_factor *= 1
        else:
            sign_factor *= 2 ** n
    
    estimated_equiv = perm_factor * sign_factor
    
    return {
        'theoretical_equiv': theoretical_equiv,
        'estimated_equiv': estimated_equiv,
        'reduction_factor': theoretical_equiv / max(estimated_equiv, 1),
        'bias_distinctness': bias_distinctness,
        'ordering_score': ordering_score,
        'is_clone_free': clones['is_clone_free'],
        'max_neuron_similarity': max_sim,
    }

# Compute metrics for all strategies
print(f"{'='*90}")
print(f"SYMMETRY-BREAKING EFFECTIVENESS")
print(f"Architecture: {train_arch} | Activation: tanh")
print(f"{'='*90}")
print()
print(f"{'Strategy':<22} {'Bias Distinct':<15} {'Ordering':<12} {'Clone-Free':<12} "
      f"{'Max Sim':<10} {'Est. Equiv':<14} {'Reduction':<12}")
print("-" * 97)

all_metrics = {}
for name, data in strategies.items():
    metrics = compute_symmetry_metrics(data['model'], train_arch)
    all_metrics[name] = metrics
    print(f"{name:<22} {metrics['bias_distinctness']:<15.3f} {metrics['ordering_score']:<12.3f} "
          f"{'Yes' if metrics['is_clone_free'] else 'No':<12} {metrics['max_neuron_similarity']:<10.3f} "
          f"{metrics['estimated_equiv']:<14,} {metrics['reduction_factor']:<12,.0f}x")

print(f"\nTheoretical equivalence classes: {all_metrics['Baseline']['theoretical_equiv']:,}")

In [ ]:
# Visualize equivalence class reduction
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

strat_names = list(all_metrics.keys())
strat_colors = [colors_strat[n] for n in strat_names]

# Bias distinctness
vals = [all_metrics[n]['bias_distinctness'] for n in strat_names]
axes[0].bar(strat_names, vals, color=strat_colors, alpha=0.8, edgecolor='black')
axes[0].set_ylabel('Fraction')
axes[0].set_title('Bias Distinctness', fontsize=12, fontweight='bold')
axes[0].set_ylim(0, 1.05)
axes[0].tick_params(axis='x', rotation=25)
axes[0].grid(True, alpha=0.3, axis='y')

# Ordering score
vals = [all_metrics[n]['ordering_score'] for n in strat_names]
axes[1].bar(strat_names, vals, color=strat_colors, alpha=0.8, edgecolor='black')
axes[1].set_ylabel('Fraction')
axes[1].set_title('Weight-Norm Ordering', fontsize=12, fontweight='bold')
axes[1].set_ylim(0, 1.05)
axes[1].tick_params(axis='x', rotation=25)
axes[1].grid(True, alpha=0.3, axis='y')

# Max neuron similarity
vals = [all_metrics[n]['max_neuron_similarity'] for n in strat_names]
axes[2].bar(strat_names, vals, color=strat_colors, alpha=0.8, edgecolor='black')
axes[2].set_ylabel('Max Cosine Similarity')
axes[2].set_title('Neuron Diversity (lower = better)', fontsize=12, fontweight='bold')
axes[2].set_ylim(0, 1.05)
axes[2].tick_params(axis='x', rotation=25)
axes[2].grid(True, alpha=0.3, axis='y')

plt.suptitle('Symmetry-Breaking Metrics Across Strategies',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Bias ordering visualization: before vs after identifiability-aware training
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (name, strat_key) in zip(axes, [
    ('Baseline', 'Baseline'),
    ('Identifiability Regularization', 'Identifiability Reg.')
]):
    layers = extract_parameters(strategies[strat_key]['model'])
    
    for l_idx, layer in enumerate(layers[:-1]):
        norms = layer['weight'].norm(dim=1).numpy()
        biases = layer['bias'].abs().numpy()
        
        x = np.arange(len(norms))
        ax.bar(x + l_idx * 0.3 - 0.15, norms, 0.3, label=f'Layer {l_idx} ||W||',
               alpha=0.7, edgecolor='black')
    
    ax.set_xlabel('Neuron Index')
    ax.set_ylabel('Weight Norm')
    ax.set_title(f'{name}\nWeight Norms per Neuron', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('Weight Norm Ordering: Symmetry-Breaking Effect',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Conclusion: Linking to Fefferman's Theorem

### Summary of Findings

**Isomorphism Detection:**
- Networks created by permuting and sign-flipping hidden neurons produce **identical outputs** (extensional isomorphism), confirming the theoretical symmetry structure.
- The Hungarian algorithm successfully recovers the underlying permutation and sign flips (faithful isomorphism), with near-zero alignment cost for true isomorphic pairs.
- Non-isomorphic networks show high alignment costs, providing a reliable discrimination criterion.

**Symmetry Breaking:**
- Standard L1/L2 regularization has **limited effect** on symmetry reduction; it primarily controls weight magnitudes.
- **Identifiability-aware regularization** (ordering + distinct biases + anti-clone) directly targets the symmetry structure:
  - Encourages distinct bias magnitudes (breaks sign-flip ambiguity)
  - Enforces weight-norm ordering (breaks permutation symmetry)
  - Penalizes clone-like neurons (prevents degenerate equivalences)

### Connection to Fefferman's Framework

Fefferman proved that for **generic** parameters (a full-measure set), the input-output map of a sigmoidal neural network determines its parameters up to the known symmetry group. Our experiments confirm:

1. **Randomly initialized** networks almost always satisfy the genericity conditions (no-clones, non-degeneracy, self-avoiding).
2. The symmetry group for tanh is $\prod_l S_{n_l} \ltimes (\mathbb{Z}/2)^{n_l}$ (permutations and sign flips), and our isomorphism detection correctly identifies these transformations.
3. Symmetry-breaking regularization can push trained networks toward a **canonical parameterization**, reducing the equivalence class size.

This has practical implications for:
- **Model comparison**: Determining if two trained networks are functionally equivalent
- **Interpretability**: Ensuring parameters can be meaningfully compared across training runs
- **Optimization landscape**: Understanding the discrete symmetries that create equivalent local minima

---

*For identifiability condition checks, see Notebook 01. For the full source code, see the `src/` directory.*